# CycloSafe - Cyclone Detection Model Training (V3)
## 100% Crash-Proof - Resumes Automatically!

**Dataset:** 5.48 Lakh NASA GIBS Satellite Images

**Architecture:** EfficientNetB0 (Transfer Learning)

**How it works:** Run ALL cells. If Colab crashes, just reconnect and Run ALL again - it picks up exactly where it left off!

## Step 1: Mount Google Drive and Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile, os, shutil, json, time, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

# Config
DRIVE_SAVE_DIR = '/content/drive/MyDrive/CycloSafe_Models'
STATE_FILE = os.path.join(DRIVE_SAVE_DIR, 'training_state.json')
BEST_MODEL_PATH = os.path.join(DRIVE_SAVE_DIR, 'best_cyclone_detector.keras')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
SEED = 42


In [ ]:
# Extract dataset (skips if already extracted)
EXTRACT_DIR = '/content/dataset_v2'
PARTS = [
    '/content/drive/MyDrive/dataset_part_1.zip',
    '/content/drive/MyDrive/dataset_part_2.zip',
    '/content/drive/MyDrive/dataset_part_3.zip',
    '/content/drive/MyDrive/dataset_part_4.zip',
]

if not os.path.exists(os.path.join(EXTRACT_DIR, 'Cyclone')):
    for i, part in enumerate(PARTS, 1):
        print(f'Extracting Part {i}/4...')
        with zipfile.ZipFile(part, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
    print('All parts extracted!')
else:
    print('Dataset already extracted.')

c_count = len(os.listdir(os.path.join(EXTRACT_DIR, 'Cyclone')))
nc_count = len(os.listdir(os.path.join(EXTRACT_DIR, 'No_Cyclone')))
print(f'Cyclone: {c_count:,} | No_Cyclone: {nc_count:,} | Total: {c_count+nc_count:,}')


## Step 2: Create Train/Val/Test Split

In [ ]:
SPLIT_DIR = '/content/dataset_split'

def create_splits(src_dir, split_dir, val_ratio=0.15, test_ratio=0.10):
    if os.path.exists(os.path.join(split_dir, 'train', 'Cyclone')):
        print('Splits already exist.')
        return
    for split in ['train', 'val', 'test']:
        for cls in ['Cyclone', 'No_Cyclone']:
            os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)
    for cls in ['Cyclone', 'No_Cyclone']:
        cls_dir = os.path.join(src_dir, cls)
        files = [f for f in os.listdir(cls_dir) if f.endswith(('.jpg','.jpeg','.png'))]
        random.seed(SEED)
        random.shuffle(files)
        n = len(files)
        n_test = int(n * test_ratio)
        n_val = int(n * val_ratio)
        n_train = n - n_test - n_val
        splits = {'train': files[:n_train], 'val': files[n_train:n_train+n_val], 'test': files[n_train+n_val:]}
        for sname, sfiles in splits.items():
            dest = os.path.join(split_dir, sname, cls)
            for f in sfiles:
                os.symlink(os.path.join(cls_dir, f), os.path.join(dest, f))
            print(f'  {cls}/{sname}: {len(sfiles):,}')

print('Creating splits...')
create_splits(EXTRACT_DIR, SPLIT_DIR)

for split in ['train', 'val', 'test']:
    c = len(os.listdir(os.path.join(SPLIT_DIR, split, 'Cyclone')))
    nc = len(os.listdir(os.path.join(SPLIT_DIR, split, 'No_Cyclone')))
    print(f'{split.upper()}: {c+nc:,} (Cyclone: {c:,} | No_Cyclone: {nc:,})')


## Step 3: Build Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0, rotation_range=30, width_shift_range=0.15,
    height_shift_range=0.15, shear_range=0.1, zoom_range=0.2,
    horizontal_flip=True, vertical_flip=True, brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_directory(os.path.join(SPLIT_DIR,'train'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True, seed=SEED)
val_gen = val_test_datagen.flow_from_directory(os.path.join(SPLIT_DIR,'val'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
test_gen = val_test_datagen.flow_from_directory(os.path.join(SPLIT_DIR,'test'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

print(f'Classes: {train_gen.class_indices}')
print(f'Train batches: {len(train_gen)} | Val: {len(val_gen)} | Test: {len(test_gen)}')


## Step 4: Build Model OR Resume from Checkpoint

In [ ]:
def build_fresh_model(input_shape=(224, 224, 3)):
    aug_model = keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ], name='augmentation')
    base_model = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False
    inputs = layers.Input(shape=input_shape, name='satellite_input')
    x = aug_model(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(0.4, name='dropout_04')(x)
    x = layers.Dense(128, activation='relu', name='dense_128')(x)
    x = layers.Dropout(0.3, name='dropout_03')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='cyclone_probability')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name='CycloSafe_Detector_V3')
    return model

# Auto-save callback - saves EVERY epoch to Drive
class DriveAutoSave(callbacks.Callback):
    def __init__(self, phase):
        self.phase = phase
        self.best_val_acc = 0.0
    def on_epoch_end(self, epoch, logs=None):
        val_acc = float(logs.get('val_accuracy', 0))
        # Save current model directly to Drive EVERY epoch
        model.save(os.path.join(DRIVE_SAVE_DIR, 'latest_model.keras'))
        # Also save as best if accuracy improved
        if val_acc > self.best_val_acc:
            self.best_val_acc = val_acc
            model.save(BEST_MODEL_PATH)
            print(f'  [AutoSave] NEW BEST model saved! val_acc: {val_acc:.4f}')
        else:
            print(f'  [AutoSave] Checkpoint saved to Drive (val_acc: {val_acc:.4f})')
        # Save training state
        state = {'phase': self.phase, 'epoch': epoch + 1,
                 'val_accuracy': val_acc,
                 'val_auc': float(logs.get('val_auc', 0)),
                 'best_val_accuracy': self.best_val_acc}
        with open(STATE_FILE, 'w') as f:
            json.dump(state, f)

# Check for existing checkpoint
RESUME_PHASE = 0
RESUME_EPOCH = 0

if os.path.exists(STATE_FILE):
    with open(STATE_FILE) as f:
        state = json.load(f)
    RESUME_PHASE = state['phase']
    RESUME_EPOCH = state['epoch']
    print(f'RESUMING from Phase {RESUME_PHASE}, Epoch {RESUME_EPOCH}')
    print(f'Last val_accuracy: {state["val_accuracy"]:.4f}')
    model = keras.models.load_model(BEST_MODEL_PATH)
    print('Model loaded from Drive!')
else:
    print('Starting fresh training...')
    model = build_fresh_model()

model.summary()


## Step 5: Phase 1 - Feature Extraction (Frozen Base)

In [ ]:
if RESUME_PHASE < 2:
    # How many epochs left in Phase 1
    PHASE1_TOTAL = 10
    start_epoch = RESUME_EPOCH if RESUME_PHASE == 1 else 0
    remaining = PHASE1_TOTAL - start_epoch

    if remaining > 0:
        model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

        phase1_cbs = [
            callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
            DriveAutoSave(phase=1),
        ]

        print(f'PHASE 1: Epochs {start_epoch+1} to {PHASE1_TOTAL}')
        t0 = time.time()
        history_p1 = model.fit(train_gen, epochs=PHASE1_TOTAL, initial_epoch=start_epoch, validation_data=val_gen, callbacks=phase1_cbs, verbose=1)
        print(f'Phase 1 done in {(time.time()-t0)/60:.1f} min')
    else:
        print('Phase 1 already completed!')
else:
    print('Phase 1 already completed in previous session!')


## Step 6: Phase 2 - Fine-Tuning (Unfrozen Base)

In [ ]:
# Unfreeze base model
for layer in model.layers:
    if layer.name == 'efficientnetb0':
        layer.trainable = True
        for sub_layer in layer.layers[:100]:
            sub_layer.trainable = False
        break

PHASE2_TOTAL = 30
start_epoch = RESUME_EPOCH if RESUME_PHASE == 2 else 0
remaining = PHASE2_TOTAL - start_epoch

if remaining > 0:
    model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    phase2_cbs = [
        callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        DriveAutoSave(phase=2),
    ]

    print(f'PHASE 2: Epochs {start_epoch+1} to {PHASE2_TOTAL}')
    t0 = time.time()
    history_p2 = model.fit(train_gen, epochs=PHASE2_TOTAL, initial_epoch=start_epoch, validation_data=val_gen, callbacks=phase2_cbs, verbose=1)
    print(f'Phase 2 done in {(time.time()-t0)/60:.1f} min')
else:
    print('Phase 2 already completed!')


## Step 7: Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns

best_model = keras.models.load_model(BEST_MODEL_PATH)
test_loss, test_accuracy, test_auc = best_model.evaluate(test_gen, verbose=1)
print(f'Test Accuracy: {test_accuracy*100:.2f}% | Test AUC: {test_auc:.4f}')

test_gen.reset()
y_pred_prob = best_model.predict(test_gen, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

print('\nCLASSIFICATION REPORT')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
roc_score = roc_auc_score(y_true, y_pred_prob)
print(f'ROC-AUC Score: {roc_score:.4f}')


## Step 8: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix - CycloSafe V3', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP: {tp:,} | TN: {tn:,} | FP: {fp:,} | FN: {fn:,}')
print(f'Precision: {tp/(tp+fp)*100:.2f}% | Recall: {tp/(tp+fn)*100:.2f}%')


## Step 9: Save Final Config

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = float(thresholds[best_idx])

config = {'threshold': best_threshold, 'test_accuracy': float(test_accuracy), 'test_auc': float(test_auc), 'roc_auc': float(roc_score), 'class_names': class_names, 'total_training_images': c_count + nc_count, 'model_version': 'V3'}
with open(os.path.join(DRIVE_SAVE_DIR, 'detection_v3_config.json'), 'w') as f:
    json.dump(config, f, indent=4)

# Clean up state file (training complete)
if os.path.exists(STATE_FILE):
    os.remove(STATE_FILE)

print('TRAINING COMPLETE!')
print(json.dumps(config, indent=4))


## Step 10: Download Model

In [ ]:
from google.colab import files
files.download(BEST_MODEL_PATH)
files.download(os.path.join(DRIVE_SAVE_DIR, 'detection_v3_config.json'))


---
## Training Complete!

**Files saved to Google Drive -> CycloSafe_Models:**
- `best_cyclone_detector.keras` - The trained model
- `detection_v3_config.json` - Config with threshold and metrics
- `confusion_matrix.png` - Visual results

**To use:** Download and place in `backend/models/`

*CycloSafe - AI-Based Tropical Cyclone Monitoring System*